In [1]:
# ─────────────────────────────────────────────────────────────
# 05 SHAP — LANL Authentication Dataset
# Explains the single-feature Isolation Forest's anomaly scores
# Note: production model uses only unique_destination_computers,
# so SHAP here confirms/quantifies this single driver per alert,
# while df retains all 10 features for rich triage context
# ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import IsolationForest

df = pd.read_csv('df_with_scores.csv')
X = pd.read_csv('features.csv')

FINAL_FEATURES = ['unique_destination_computers']
FINAL_CONTAMINATION = 0.15

print(f"Loaded {len(df):,} events")
print(f"Anomalies flagged: {(df['if_prediction']==-1).sum():,}")

# Rebuild the exact production model for SHAP explainer
model = IsolationForest(n_estimators=200, contamination=FINAL_CONTAMINATION, random_state=42)
model.fit(X[FINAL_FEATURES])

print(f"\nModel rebuilt for SHAP — using feature: {FINAL_FEATURES}")

c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 68,221 events
Anomalies flagged: 9,581

Model rebuilt for SHAP — using feature: ['unique_destination_computers']


In [ ]:
# Rebuild the standard model (no need for max_features workaround)
model = IsolationForest(
    n_estimators=200,
    contamination=FINAL_CONTAMINATION,
    random_state=42
)
model.fit(X[FINAL_FEATURES])

# KernelExplainer is model-agnostic — works with any prediction function,
# sidesteps TreeExplainer's internal tree-traversal bug entirely
background = shap.sample(X[FINAL_FEATURES], 100, random_state=42)

def if_score_func(X_input):
    return model.decision_function(X_input)

explainer = shap.KernelExplainer(if_score_func, background)

print("KernelExplainer created — computing SHAP values for all events...")
print("(this will take longer than TreeExplainer, but is reliable for single-feature models)")

shap_values = explainer.shap_values(X[FINAL_FEATURES], nsamples=100)

print(f"\nSHAP values computed for {len(X):,} events")
print(f"Shape: {shap_values.shape}")

In [1]:
# Only compute SHAP for flagged anomalies — this is all we need
# for triage cards, and is dramatically faster than all 68,221 events

anomalies_only = X[FINAL_FEATURES][df['if_prediction'] == -1]
print(f"Computing SHAP for {len(anomalies_only):,} flagged anomalies only (not all 68,221 events)")

background = shap.sample(X[FINAL_FEATURES], 50, random_state=42)  # smaller background too

def if_score_func(X_input):
    return model.decision_function(X_input)

explainer = shap.KernelExplainer(if_score_func, background)

print("Computing SHAP values...")
shap_values_anomalies = explainer.shap_values(anomalies_only, nsamples=50)

print(f"\nDone — SHAP values computed for {len(shap_values_anomalies):,} anomalies")
print(f"Shape: {shap_values_anomalies.shape}")

NameError: name 'X' is not defined

In [2]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import IsolationForest

df = pd.read_csv('df_with_scores.csv')
X = pd.read_csv('features.csv')

FINAL_FEATURES = ['unique_destination_computers']
FINAL_CONTAMINATION = 0.15

model = IsolationForest(n_estimators=200, contamination=FINAL_CONTAMINATION, random_state=42)
model.fit(X[FINAL_FEATURES])

print(f"Loaded {len(df):,} events, model rebuilt")

c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 68,221 events, model rebuilt


In [3]:
anomalies_only = X[FINAL_FEATURES][df['if_prediction'] == -1]
print(f"Computing SHAP for {len(anomalies_only):,} flagged anomalies only")

background = shap.sample(X[FINAL_FEATURES], 30, random_state=42)

def if_score_func(X_input):
    return model.decision_function(X_input)

explainer = shap.KernelExplainer(if_score_func, background)

print("Computing SHAP values (this should take under a minute)...")
shap_values_anomalies = explainer.shap_values(anomalies_only, nsamples=30)

print(f"\nDone — SHAP values computed for {len(shap_values_anomalies):,} anomalies")

c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(


Computing SHAP for 9,581 flagged anomalies only
Computing SHAP values (this should take under a minute)...


  0%|          | 0/9581 [00:00<?, ?it/s]c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but IsolationForest was fitted with feature names
  warnings.warn(
c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-


Done — SHAP values computed for 9,581 anomalies


In [4]:
print(f"SHAP values shape: {shap_values_anomalies.shape}")
print(f"SHAP value range: {shap_values_anomalies.min():.4f} to {shap_values_anomalies.max():.4f}")
print(f"Mean SHAP value: {shap_values_anomalies.mean():.4f}")

# Attach back to the anomalies dataframe for inspection
anomalies_df = df[df['if_prediction'] == -1].copy()
anomalies_df['shap_value'] = shap_values_anomalies.flatten()

print(f"\nVerification — SHAP value should align closely with if_score:")
print(anomalies_df[['if_score', 'shap_value', 'unique_destination_computers', 'is_attack']].head(10))

correlation = anomalies_df['if_score'].corr(anomalies_df['shap_value'])
print(f"\nCorrelation between if_score and shap_value: {correlation:.4f}")

SHAP values shape: (9581, 1)
SHAP value range: -0.3612 to -0.0858
Mean SHAP value: -0.1319

Verification — SHAP value should align closely with if_score:
      if_score  shap_value  unique_destination_computers  is_attack
2112 -0.013429   -0.091047                             7          0
2113 -0.013429   -0.091047                             7          0
2114 -0.013429   -0.091047                             7          0
2115 -0.013429   -0.091047                             7          0
2116 -0.013429   -0.091047                             7          0
2117 -0.013429   -0.091047                             7          0
2118 -0.013429   -0.091047                             7          0
2119 -0.013429   -0.091047                             7          0
2492 -0.013429   -0.091047                             7          0
2493 -0.013429   -0.091047                             7          0

Correlation between if_score and shap_value: 1.0000


In [5]:
# Save SHAP values alongside the anomalies for use in triage cards
anomalies_df.to_csv('shap_anomalies.csv', index=False)
print(f"Saved shap_anomalies.csv — {len(anomalies_df):,} rows")

print(f"\nSample of most anomalous events (lowest if_score):")
top_anomalies = anomalies_df.nsmallest(10, 'if_score')
print(top_anomalies[['source_user', 'destination_computer', 'if_score', 
                      'shap_value', 'unique_destination_computers', 'is_attack']])

Saved shap_anomalies.csv — 9,581 rows

Sample of most anomalous events (lowest if_score):
      source_user destination_computer  if_score  shap_value  \
46205  U1653@DOM1                 C754 -0.283586   -0.361203   
46206  U1653@DOM1                 C754 -0.283586   -0.361203   
46207  U1653@DOM1                 C754 -0.283586   -0.361203   
46208  U1653@DOM1                C9378 -0.283586   -0.361203   
46209  U1653@DOM1                 C754 -0.283586   -0.361203   
46210  U1653@DOM1                C9608 -0.283586   -0.361203   
46211  U1653@DOM1                 C754 -0.283586   -0.361203   
46212  U1653@DOM1                 C754 -0.283586   -0.361203   
46213  U1653@DOM1               C13496 -0.283586   -0.361203   
46214  U1653@DOM1               C16732 -0.283586   -0.361203   

       unique_destination_computers  is_attack  
46205                            66          1  
46206                            66          1  
46207                            66          1  
46208    

In [6]:
# ─────────────────────────────────────────────────────────────
# MITRE ATT&CK Mapping Engine — LANL Dataset
# Grounded in the statistically validated attack signature:
# lateral movement breadth is the primary indicator (T1021)
# ─────────────────────────────────────────────────────────────

def map_to_attck(row):
    """
    Maps anomalies to ATT&CK techniques based on the validated
    feature evidence from this dataset's statistical analysis.
    """
    dest_count = row['unique_destination_computers']

    if dest_count >= 30:
        return {
            'technique_id': 'T1021',
            'technique_name': 'Remote Services',
            'tactic': 'Lateral Movement',
            'confidence': 'HIGH',
            'action': 'Escalate immediately. Audit all destination computers for signs of compromise. Reset credentials.'
        }
    elif dest_count >= 20:
        return {
            'technique_id': 'T1021',
            'technique_name': 'Remote Services',
            'tactic': 'Lateral Movement',
            'confidence': 'HIGH',
            'action': 'Review authentication pattern across destinations. Check for credential reuse.'
        }
    elif dest_count >= 15:
        return {
            'technique_id': 'T1078',
            'technique_name': 'Valid Accounts',
            'tactic': 'Defense Evasion / Persistence',
            'confidence': 'MEDIUM',
            'action': 'Monitor for continued spread. Correlate with logon_type for RDP/scripted activity.'
        }
    else:
        return {
            'technique_id': 'T1078',
            'technique_name': 'Valid Accounts',
            'tactic': 'Defense Evasion',
            'confidence': 'LOW',
            'action': 'Low-confidence anomaly. Monitor for escalation in lateral movement breadth.'
        }

attck_results = anomalies_df.apply(map_to_attck, axis=1, result_type='expand')
anomalies_df = pd.concat([anomalies_df, attck_results], axis=1)

print("ATT&CK Attribution Results:")
print(f"\nTechnique distribution:")
print(anomalies_df['technique_id'].value_counts())
print(f"\nConfidence distribution:")
print(anomalies_df['confidence'].value_counts())

print(f"\nAccuracy check — does confidence correlate with actual attack status?")
for conf in ['HIGH', 'MEDIUM', 'LOW']:
    subset = anomalies_df[anomalies_df['confidence'] == conf]
    if len(subset) > 0:
        precision = subset['is_attack'].mean()
        print(f"  {conf}: {len(subset):,} alerts, {precision*100:.1f}% genuine attacks")

anomalies_df.to_csv('attack_tagged_alerts.csv', index=False)
print(f"\nSaved attack_tagged_alerts.csv")

ATT&CK Attribution Results:

Technique distribution:
technique_id
T1021    5564
T1078    4017
Name: count, dtype: int64

Confidence distribution:
confidence
HIGH    5564
LOW     4017
Name: count, dtype: int64

Accuracy check — does confidence correlate with actual attack status?
  HIGH: 5,564 alerts, 93.5% genuine attacks
  LOW: 4,017 alerts, 39.8% genuine attacks

Saved attack_tagged_alerts.csv


In [7]:
def generate_triage_card(row):
    card = f"""
{'='*70}
TRIAGE CARD — {row['confidence']} CONFIDENCE
{'='*70}
User: {row['source_user']}
Destination: {row['destination_computer']}
Time: {row['time']}

DETECTION
  IF Anomaly Score: {row['if_score']:.4f}
  Primary Driver (SHAP): unique_destination_computers = {row['unique_destination_computers']}
  SHAP Value: {row['shap_value']:.4f} (100% of anomaly score — single-feature model)

MITRE ATT&CK
  Technique: {row['technique_id']} — {row['technique_name']}
  Tactic: {row['tactic']}
  Confidence: {row['confidence']}

RECOMMENDED ACTION
  {row['action']}
{'='*70}
"""
    return card

# Show example cards — one HIGH confidence, one LOW confidence
print("EXAMPLE — HIGH CONFIDENCE CARD:")
high_example = anomalies_df[anomalies_df['confidence']=='HIGH'].iloc[0]
print(generate_triage_card(high_example))

print("\nEXAMPLE — LOW CONFIDENCE CARD:")
low_example = anomalies_df[anomalies_df['confidence']=='LOW'].iloc[0]
print(generate_triage_card(low_example))

EXAMPLE — HIGH CONFIDENCE CARD:

TRIAGE CARD — HIGH CONFIDENCE
User: C1085$@DOM1
Destination: C1708
Time: 1036889

DETECTION
  IF Anomaly Score: -0.0082
  Primary Driver (SHAP): unique_destination_computers = 21
  SHAP Value: -0.0858 (100% of anomaly score — single-feature model)

MITRE ATT&CK
  Technique: T1021 — Remote Services
  Tactic: Lateral Movement
  Confidence: HIGH

RECOMMENDED ACTION
  Review authentication pattern across destinations. Check for credential reuse.


EXAMPLE — LOW CONFIDENCE CARD:

TRIAGE CARD — LOW CONFIDENCE
User: C10$@DOM1
Destination: C3374
Time: 1059081

DETECTION
  IF Anomaly Score: -0.0134
  Primary Driver (SHAP): unique_destination_computers = 7
  SHAP Value: -0.0910 (100% of anomaly score — single-feature model)

MITRE ATT&CK
  Technique: T1078 — Valid Accounts
  Tactic: Defense Evasion
  Confidence: LOW

RECOMMENDED ACTION
  Low-confidence anomaly. Monitor for escalation in lateral movement breadth.



In [8]:
print("Distribution of unique_destination_computers among ALL flagged anomalies:")
print(anomalies_df['unique_destination_computers'].describe())

print(f"\nHow many anomalies have unique_destination_computers < 15 (our LOW threshold)?")
low_dest_anomalies = anomalies_df[anomalies_df['unique_destination_computers'] < 15]
print(f"Count: {len(low_dest_anomalies):,}")
print(f"Of these, genuine attacks: {low_dest_anomalies['is_attack'].sum():,} ({low_dest_anomalies['is_attack'].mean()*100:.1f}%)")

print(f"\nWhy would the IF flag low-destination-count events as anomalies?")
print(f"Min if_score for events with dest_count < 15: {low_dest_anomalies['if_score'].min():.4f}")
print(f"Max if_score for events with dest_count < 15: {low_dest_anomalies['if_score'].max():.4f}")
print(f"\nMin if_score for events with dest_count >= 20: {anomalies_df[anomalies_df['unique_destination_computers']>=20]['if_score'].min():.4f}")
print(f"Max if_score for events with dest_count >= 20: {anomalies_df[anomalies_df['unique_destination_computers']>=20]['if_score'].max():.4f}")

Distribution of unique_destination_computers among ALL flagged anomalies:
count    9581.000000
mean       19.195700
std        11.139437
min         7.000000
25%        10.000000
50%        20.000000
75%        25.000000
max        66.000000
Name: unique_destination_computers, dtype: float64

How many anomalies have unique_destination_computers < 15 (our LOW threshold)?
Count: 4,017
Of these, genuine attacks: 1,600 (39.8%)

Why would the IF flag low-destination-count events as anomalies?
Min if_score for events with dest_count < 15: -0.0735
Max if_score for events with dest_count < 15: -0.0134

Min if_score for events with dest_count >= 20: -0.2836
Max if_score for events with dest_count >= 20: -0.0082


In [9]:
def map_to_attck(row):
    dest_count = row['unique_destination_computers']

    if dest_count >= 30:
        confidence, action = 'CRITICAL', 'Escalate immediately. Audit all destination computers for signs of compromise. Reset credentials.'
    elif dest_count >= 20:
        confidence, action = 'HIGH', 'Review authentication pattern across destinations. Check for credential reuse.'
    elif dest_count >= 15:
        confidence, action = 'MEDIUM', 'Monitor for continued spread. Correlate with logon_type for RDP/scripted activity.'
    else:
        confidence, action = 'MEDIUM-LOW', 'Elevated lateral movement (above baseline). Batch review recommended, not urgent.'

    return {
        'technique_id': 'T1021' if dest_count >= 20 else 'T1078',
        'technique_name': 'Remote Services' if dest_count >= 20 else 'Valid Accounts',
        'tactic': 'Lateral Movement' if dest_count >= 20 else 'Defense Evasion',
        'confidence': confidence,
        'action': action
    }

attck_results = anomalies_df.apply(map_to_attck, axis=1, result_type='expand')
anomalies_df = pd.concat([anomalies_df.drop(columns=['technique_id','technique_name','tactic','confidence','action'], errors='ignore'), attck_results], axis=1)

print("Updated confidence distribution:")
print(anomalies_df['confidence'].value_counts())
for conf in anomalies_df['confidence'].unique():
    subset = anomalies_df[anomalies_df['confidence'] == conf]
    print(f"  {conf}: {len(subset):,} alerts, {subset['is_attack'].mean()*100:.1f}% genuine attacks")

anomalies_df.to_csv('attack_tagged_alerts.csv', index=False)
print(f"\nSaved attack_tagged_alerts.csv (updated)")

Updated confidence distribution:
confidence
HIGH          4537
MEDIUM-LOW    4017
CRITICAL      1027
Name: count, dtype: int64
  MEDIUM-LOW: 4,017 alerts, 39.8% genuine attacks
  HIGH: 4,537 alerts, 97.0% genuine attacks
  CRITICAL: 1,027 alerts, 77.9% genuine attacks

Saved attack_tagged_alerts.csv (updated)


In [10]:
critical_tier = anomalies_df[anomalies_df['confidence'] == 'CRITICAL']
high_tier = anomalies_df[anomalies_df['confidence'] == 'HIGH']

print("CRITICAL tier — who are the FALSE positives (not actual attacks)?")
critical_fp = critical_tier[critical_tier['is_attack'] == 0]
print(f"Count: {len(critical_fp):,}")
print(f"\nDistinct users in CRITICAL false positives:")
print(critical_fp['source_user'].value_counts().head(10))

print(f"\nTheir unique_destination_computers values:")
print(critical_fp['unique_destination_computers'].describe())

CRITICAL tier — who are the FALSE positives (not actual attacks)?
Count: 227

Distinct users in CRITICAL false positives:
source_user
U292@DOM1    227
Name: count, dtype: int64

Their unique_destination_computers values:
count    227.0
mean      30.0
std        0.0
min       30.0
25%       30.0
50%       30.0
75%       30.0
max       30.0
Name: unique_destination_computers, dtype: float64


In [11]:
u292_events = df[df['source_user'] == 'U292@DOM1']
print(f"Total events for U292@DOM1: {len(u292_events)}")
print(f"\nAuthentication type distribution:")
print(u292_events['authentication_type'].value_counts())
print(f"\nLogon type distribution:")
print(u292_events['logon_type'].value_counts())
print(f"\nSuccess/failure breakdown:")
print(u292_events['success_failure'].value_counts())
print(f"\nDestination computers touched:")
print(u292_events['destination_computer'].nunique())

Total events for U292@DOM1: 227

Authentication type distribution:
authentication_type
?           224
Kerberos      3
Name: count, dtype: int64

Logon type distribution:
logon_type
?                   215
Network               9
NetworkCleartext      3
Name: count, dtype: int64

Success/failure breakdown:
success_failure
Success    227
Name: count, dtype: int64

Destination computers touched:
30


In [12]:
print("="*60)
print("  05_SHAP.IPYNB — SUMMARY")
print("="*60)
print(f"\nSHAP values computed for {len(anomalies_df):,} flagged anomalies")
print(f"Correlation between if_score and shap_value: 1.0000 (single-feature model)")
print(f"\nATT&CK Confidence Tier Distribution:")
for conf in ['CRITICAL', 'HIGH', 'MEDIUM-LOW']:
    subset = anomalies_df[anomalies_df['confidence'] == conf]
    if len(subset) > 0:
        print(f"  {conf:<12}: {len(subset):>5,} alerts, {subset['is_attack'].mean()*100:>5.1f}% genuine attacks")
print(f"\nKnown limitation: U292@DOM1 (227 events) reduces CRITICAL precision")
print(f"  due to legitimate high-volume automated authentication")
print(f"\nSaved: shap_anomalies.csv, attack_tagged_alerts.csv")

  05_SHAP.IPYNB — SUMMARY

SHAP values computed for 9,581 flagged anomalies
Correlation between if_score and shap_value: 1.0000 (single-feature model)

ATT&CK Confidence Tier Distribution:
  CRITICAL    : 1,027 alerts,  77.9% genuine attacks
  HIGH        : 4,537 alerts,  97.0% genuine attacks
  MEDIUM-LOW  : 4,017 alerts,  39.8% genuine attacks

Known limitation: U292@DOM1 (227 events) reduces CRITICAL precision
  due to legitimate high-volume automated authentication

Saved: shap_anomalies.csv, attack_tagged_alerts.csv


In [1]:
import pandas as pd
import numpy as np
import shap
from sklearn.ensemble import IsolationForest

# Load the user-day data and production model configuration
df = pd.read_csv('df_with_scores.csv')
X = pd.read_csv('features.csv')

FINAL_FEATURES = X.columns.tolist()
FINAL_CONTAMINATION = 0.012

# Rebuild production model
model = IsolationForest(n_estimators=200, contamination=FINAL_CONTAMINATION,
                        random_state=42)
model.fit(X)

print(f"Production model rebuilt: {len(FINAL_FEATURES)} features")
print(f"Features: {FINAL_FEATURES}")

# ── Try TreeExplainer first on the 12-feature model ──────────
print("\nAttempting TreeExplainer on 12-feature model...")
try:
    explainer_tree = shap.TreeExplainer(model)
    # Test on a small sample first
    test_sample = X.iloc[:5]
    shap_test = explainer_tree.shap_values(test_sample)
    print(f"TreeExplainer SUCCESS — shape: {shap_test.shape}")
    USE_TREE = True
except Exception as e:
    print(f"TreeExplainer FAILED: {type(e).__name__}: {str(e)[:100]}")
    USE_TREE = False
    print("Will use KernelExplainer instead")

c:\Users\user\Desktop\auth_anomaly_detection_LANL\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Production model rebuilt: 12 features
Features: ['total_logons', 'failed_logons', 'failure_rate', 'unique_destinations', 'unique_source_computers', 'unique_logon_types', 'batch_or_rdp_logons', 'network_logons', 'unique_auth_types', 'kerberos_logons', 'ntlm_logons', 'active_hours']

Attempting TreeExplainer on 12-feature model...
TreeExplainer SUCCESS — shape: (5, 12)


In [2]:
import time

# ── Full SHAP computation using TreeExplainer ────────────────
print("Computing TreeSHAP values for all 14,416 user-day profiles...")
print("(TreeExplainer is fast — should complete in seconds)\n")

start = time.perf_counter()
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X)
elapsed = time.perf_counter() - start

print(f"SHAP values computed in {elapsed:.2f} seconds")
print(f"Shape: {shap_values.shape}  (14,416 user-days × 12 features)")
print(f"SHAP value range: {shap_values.min():.4f} to {shap_values.max():.4f}")

# ── Per-feature mean absolute SHAP value ─────────────────────
mean_abs_shap = np.abs(shap_values).mean(axis=0)
feature_importance = pd.DataFrame({
    'feature': FINAL_FEATURES,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False)

print(f"\nFeature importance (mean |SHAP value|):")
print(feature_importance.to_string(index=False))

Computing TreeSHAP values for all 14,416 user-day profiles...
(TreeExplainer is fast — should complete in seconds)

SHAP values computed in 45.93 seconds
Shape: (14416, 12)  (14,416 user-days × 12 features)
SHAP value range: -3.6602 to 0.5286

Feature importance (mean |SHAP value|):
                feature  mean_abs_shap
     unique_logon_types       0.333431
            ntlm_logons       0.292054
      unique_auth_types       0.255914
unique_source_computers       0.233561
    unique_destinations       0.224922
           total_logons       0.205520
        kerberos_logons       0.191835
           active_hours       0.181865
         network_logons       0.181777
          failed_logons       0.050936
           failure_rate       0.050153
    batch_or_rdp_logons       0.014260


In [3]:
import pandas as pd
import numpy as np
import shap
import time
from sklearn.ensemble import IsolationForest

# Confirm everything needed is available
print(f"X shape: {X.shape}")
print(f"FINAL_FEATURES: {FINAL_FEATURES}")
print(f"model type: {type(model)}")
print(f"explainer type: {type(explainer)}")
print(f"shap_values shape: {shap_values.shape}")
print(f"df shape: {df.shape}")
print(f"if_prediction column exists: {'if_prediction' in df.columns}")

X shape: (14416, 12)
FINAL_FEATURES: ['total_logons', 'failed_logons', 'failure_rate', 'unique_destinations', 'unique_source_computers', 'unique_logon_types', 'batch_or_rdp_logons', 'network_logons', 'unique_auth_types', 'kerberos_logons', 'ntlm_logons', 'active_hours']
model type: <class 'sklearn.ensemble._iforest.IsolationForest'>
explainer type: <class 'shap.explainers._tree.TreeExplainer'>
shap_values shape: (14416, 12)
df shape: (14416, 19)
if_prediction column exists: True


In [4]:
# ── Explanation Generation Time ───────────────────────────────
print("Measuring explanation generation time...\n")

single_alert = X.iloc[[0]]
start = time.perf_counter()
_ = explainer.shap_values(single_alert)
end = time.perf_counter()
single_time_ms = (end - start) * 1000
print(f"Single alert: {single_time_ms:.2f} ms")

batch_alerts = X.iloc[:10]
start = time.perf_counter()
_ = explainer.shap_values(batch_alerts)
end = time.perf_counter()
batch_time_ms = (end - start) * 1000 / 10
print(f"Batch avg (10 alerts): {batch_time_ms:.2f} ms per alert")

# ── Explanation Consistency — 5 repeated runs ─────────────────
print("\nTesting explanation consistency (5 repeated runs on same 10 alerts)...")
test_alerts = X.iloc[:10]
repeated_runs = []
for run in range(5):
    vals = explainer.shap_values(test_alerts)
    repeated_runs.append(vals)

repeated_runs = np.array(repeated_runs)

std_per_alert = repeated_runs.std(axis=0)
mean_per_alert = np.abs(repeated_runs.mean(axis=0))
cv_per_alert = np.where(mean_per_alert > 0, std_per_alert / mean_per_alert, 0)

print(f"Mean CV across all alerts and features: {cv_per_alert.mean():.6f}")
print(f"Max CV: {cv_per_alert.max():.6f}")

# ── Top-k Feature Stability ───────────────────────────────────
print("\nTop-3 feature stability across 5 repeated runs:")
for run_idx in range(5):
    run_importance = np.abs(repeated_runs[run_idx]).mean(axis=0)
    top3 = pd.Series(run_importance, index=FINAL_FEATURES).nlargest(3)
    print(f"  Run {run_idx+1}: {list(top3.index)}")

# ── SHAP for flagged anomalies only ──────────────────────────
anomaly_mask = (df['if_prediction'] == -1).values
shap_anomalies = shap_values[anomaly_mask]
X_anomalies = X[anomaly_mask].copy()

print(f"\nSHAP computed for {shap_anomalies.shape[0]:,} flagged anomalies")
print(f"Features per anomaly: {shap_anomalies.shape[1]}")

df_anomalies = df[anomaly_mask].copy()
for i, feat in enumerate(FINAL_FEATURES):
    df_anomalies[f'shap_{feat}'] = shap_anomalies[:, i]

df_anomalies['shap_top_feature'] = [
    FINAL_FEATURES[np.abs(shap_anomalies[i]).argmax()]
    for i in range(len(shap_anomalies))
]

df_anomalies.to_csv('shap_anomalies.csv', index=False)
feature_importance.to_csv('shap_feature_importance.csv', index=False)

print(f"\nSaved shap_anomalies.csv")
print(f"Saved shap_feature_importance.csv")

print(f"\n{'='*55}")
print(f"  PHASE 3 SUMMARY — TreeSHAP (12-feature model)")
print(f"{'='*55}")
print(f"  Explainer:               TreeExplainer (TreeSHAP)")
print(f"  Single alert latency:    {single_time_ms:.2f} ms")
print(f"  Batch avg latency:       {batch_time_ms:.2f} ms")
print(f"  Consistency (CV):        {cv_per_alert.mean():.6f}")
print(f"  Max CV:                  {cv_per_alert.max():.6f}")
print(f"  Explanation coverage:    100% ({shap_anomalies.shape[0]:,} anomalies)")
print(f"  Top feature (global):    {feature_importance.iloc[0]['feature']}")
print(f"  Features explained:      {len(FINAL_FEATURES)}")

Measuring explanation generation time...

Single alert: 5.71 ms
Batch avg (10 alerts): 4.72 ms per alert

Testing explanation consistency (5 repeated runs on same 10 alerts)...
Mean CV across all alerts and features: 0.000000
Max CV: 0.000000

Top-3 feature stability across 5 repeated runs:
  Run 1: ['ntlm_logons', 'network_logons', 'total_logons']
  Run 2: ['ntlm_logons', 'network_logons', 'total_logons']
  Run 3: ['ntlm_logons', 'network_logons', 'total_logons']
  Run 4: ['ntlm_logons', 'network_logons', 'total_logons']
  Run 5: ['ntlm_logons', 'network_logons', 'total_logons']

SHAP computed for 173 flagged anomalies
Features per anomaly: 12

Saved shap_anomalies.csv
Saved shap_feature_importance.csv

  PHASE 3 SUMMARY — TreeSHAP (12-feature model)
  Explainer:               TreeExplainer (TreeSHAP)
  Single alert latency:    5.71 ms
  Batch avg latency:       4.72 ms
  Consistency (CV):        0.000000
  Max CV:                  0.000000
  Explanation coverage:    100% (173 anomali